# Reto 01: Expected Credit Loss — Post-Challenge Analysis
## ¿Qué mantuve, qué corregí y qué aprendí del reto?

Este notebook es una segunda versión de mi solución del reto de Pérdida Esperada, despues de conocer los resultados, revisar el código de los otros cracks del reto, del jefe y mi feedback realicé esta versión.  No reemplaza la entrega original ni pretende presentar retrospectivamente una estrategia que no desarrollé durante mi primera versión. Lo utilizo para documentar qué funcionó, dónde estuvo el principal costo de mi primera solución y cómo cambié la política de margen después de recibir feedback.

La estructura sigue la lógica que utilicé en el proyecto original, con preguntas, evidenicia, interpretaciones y decisiones. Los comentarios dentro del código explican qué hace cada bloque y por qué se programa de esa forma, incorporando esta mejora también. Y, los textos fuera del código se utilizan para interpretar resultados, justificar decisiones y dejar claras las limitaciones.

La pérdida esperada se mantiene como:

$$ECL_i = PD_i \times LGD_i \times EAD_i$$

pero el aprendizaje principal de este análisis está en **cómo asignar el margen conservador para cumplir las restricciones sin elevar innecesariamente el ECL de toda la cartera**. Es decir, cómo optimizar el ECL.

En esta versión separo dos resultados:

- **V2A** es mi propuesta principal del reto. Mantiene el ECL basal de las operaciones no cubiertas y cambia la forma de priorizar y dimensionar el margen.
- **V2B** es un experimento adicional con $\kappa$, incorporado después de revisar las soluciones compartidas por otros participantes y la solución de referencia de Jyts. Sirve para entender mejor la función objetivo del reto, pero no lo presento como una política que trasladaría directamente a un banco.

## 1. Punto de partida: ¿qué pasó en mi primera versión (V1)?

Con el split oficial del reto, mi solución terminó como quinta en ECL total. El resultado reportado fue aproximadamente:

| Métrica | Resultado oficial V1 |
|---|---:|
| ECL total | USD 1,053.7 M |
| ECL / pérdida real | 1.90x |
| Préstamos con ECL < pérdida real | 9.84% |
| MAE | USD 2,668 |
| MAPE | 83.1% |

El feedback que recibí fue bastante claro. El EDA y la disciplina Train/Test estaban bien, la PD con WoE + regresión logística también estaba bien calibrada. Pero el exceso en el ECL venía del margen, porque yo cubría aproximadamente el tramo de mayor PD con un piso cercano al 90% del importe financiado. La observación que más cambió mi forma de plantear el problema fue esta pregunta:

> **¿Qué es lo mínimo que necesita cada préstamo para quedar cubierto?**

En la V1 respondí de forma demasiado amplia utilizando `funded_amnt`. En esta segunda iteración utilizo la pérdida condicionada estimada y, después, el costo incremental de llevar una operación desde su ECL basal hasta el nivel de cobertura.


## 2. ¿Qué cambio y qué no cambio?

No quiero que la mejora venga de flexibilizar las reglas de la primera versión. Por eso, mantengo estas condiciones:

- una sola variable `SEED` controla todas las fuentes de aleatoriedad
- el split oficial es aleatorio, 20% de Test y sin estratificación
- Test no participa en limpieza, ingeniería de variables, selección, entrenamiento ni calibración
- `grade` y las variables de desempeño no entran al modelo de PD
- EAD y LGD se estiman únicamente utilizando defaults de Train
- la pérdida real del Test se reconstruye solo cuando el pipeline ya está congelado


Lo que sí corrijo respecto a mi primera versión es:

- uso exactamente el split oficial utilizado para la evaluación, `SEED = 1710`
- mantengo EAD en escala relativa, porque permite modelar la exposición independientemente del tamaño nominal del préstamo
- calculo el MAPE únicamente sobre operaciones con pérdida real positiva, siguiendo la definición utilizada en la evaluación
- cambio la forma de asignar el margen conservador, dejo de priorizar únicamente por PD y paso a considerar la pérdida condicionada al default y el costo incremental de cobertura.

No recalibro PD, EAD o LGD utilizando Test. El principal cambio de esta segunda versión está en la política que utiliza sus predicciones para construir el ECL conservador.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from lightgbm import LGBMRegressor

# Una sola semilla controla todo el pipeline.
SEED = 1710
TEST_SIZE = 0.20
VAL_SIZE = 0.20       # 20% de lo que queda después de separar Test
N_BOOT = 300

np.random.seed(SEED)
rng = np.random.default_rng(SEED)

DATA_DIR = Path('./Data')
assert (DATA_DIR / 'X.csv').exists(), 'No encuentro X.csv en la carpeta de datos del notebook.'
assert (DATA_DIR / 'target.csv').exists(), 'No encuentro target.csv en la carpeta de datos del notebook.'


## 3. Carga y split oficial

Primero separo Test. No consulto sus métricas para decidir nada. Después divido el 80% restante en Fit y Validación. Esta estructura deja aproximadamente 64% para ajuste, 16% para validación y 20% para Test. No utilizo stratify porque quiero reproducir exactamente el split oficial con el que se evaluaron todos los notebooks.


In [3]:
model_cols = [
    'funded_amnt', 'interest_rate', 'monthly_payment', 'emp_length',
    'home_ownership_status', 'annual_income', 'verification_status',
    'loan_purpose', 'initial_list_status', 'disbursement_method',
    'loan_term_months', 'issue_date_month', 'issue_date_year', 'region_code',
    'earliest_cr_line_year', 'dept_paym_income_ratio', 'used_credit_share',
    'num_inq_in_6mths', 'num_30+_delinq_in_2yrs', 'num_derogatory_pub_rec',
    'total_credit_revolving_bal',
    # Las siguientes variables son de desempeño. Se cargan para construir
    # EAD, LGD y pérdida real; nunca pasan al modelo de PD.
    'paym_rec_for_tot_amnt_fund', 'princ_rec', 'interest_rec', 'late_fees_rec'
]

X = pd.read_csv(DATA_DIR / 'X.csv', usecols=model_cols)
y = pd.read_csv(DATA_DIR / 'target.csv')['y'].astype('int8')

indices = np.arange(len(X))
rest_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=SEED
)
fit_idx, val_idx = train_test_split(
    rest_idx,
    test_size=VAL_SIZE,
    random_state=SEED
)

fit_idx = np.sort(fit_idx)
val_idx = np.sort(val_idx)
test_idx = np.sort(test_idx)

print(f'Fit: {len(fit_idx):,}')
print(f'Validación: {len(val_idx):,}')
print(f'Test: {len(test_idx):,}')


Fit: 1,369,371
Validación: 342,343
Test: 427,929


El tamaño de las muestras coincide con el esquema utilizado en la solución de referencia. Fit concentra la mayor parte de la información, Validación se reserva para decisiones de modelización y margen, y Test permanece cerrado. El hecho de que las tasas de default sean similares entre conjuntos es una consecuencia del tamaño de la cartera, no de haber forzado una estratificación.

## 4. Reconstrucción de EAD, LGD y pérdida real

Las variables posteriores a originación se mantienen separadas del PD. Su función aquí es construir los targets observados.

Para una operación en default:

$$EAD = funded\_amnt - princ\_rec$$

$$Recuperaciones = paym\_rec - princ\_rec - interest\_rec - late\_fees\_rec$$

$$LGD = 1 - \frac{Recuperaciones}{EAD}$$

$$Pérdida\ real = EAD \times LGD$$

Para una operación sana, la pérdida real es cero.


In [4]:
# Estos targets se construyen para todas las filas, pero EAD y LGD solo se utilizan como targets de modelización en defaults de Fit.
recovery = (
    X['paym_rec_for_tot_amnt_fund']
    - X['princ_rec']
    - X['interest_rec']
    - X['late_fees_rec']
).clip(lower=0)

ead_obs = (X['funded_amnt'] - X['princ_rec']).clip(lower=0)

ead_ratio = (
    ead_obs / X['funded_amnt'].replace(0, np.nan)
).clip(0, 1)

lgd_obs = pd.Series(
    np.where(
        ead_obs > 0,
        1 - recovery / ead_obs,
        np.nan
    ),
    index=X.index
).clip(0, 1)

loss_real = np.where(
    y.to_numpy() == 1,
    ead_obs.to_numpy() * lgd_obs.fillna(0).to_numpy(),
    0.0
)


## 5. Feature engineering de originación

En esta V2 no intento rehacer todo el modelo. Mantengo la lógica de la primera entrega porque el feedback mostró que el problema principal no estaba allí. Agrego únicamente variables derivadas que pueden construirse con información de originación y que ya tenían una interpretación clara en el EDA, como lo son: antigüedad crediticia, tamaño del préstamo respecto al ingreso y presión de la cuota.


In [5]:
def construir_features(df):
    cols = [
        'funded_amnt', 'interest_rate', 'monthly_payment', 'emp_length',
        'home_ownership_status', 'annual_income', 'verification_status',
        'loan_purpose', 'initial_list_status', 'disbursement_method',
        'loan_term_months', 'issue_date_month', 'issue_date_year', 'region_code',
        'earliest_cr_line_year', 'dept_paym_income_ratio', 'used_credit_share',
        'num_inq_in_6mths', 'num_30+_delinq_in_2yrs', 'num_derogatory_pub_rec',
        'total_credit_revolving_bal'
    ]
    out = df[cols].copy()

    # Antigüedad del historial crediticio.
    out['credit_history_years'] = (
        out['issue_date_year'] - out['earliest_cr_line_year']
    )

    # Relaciono el tamaño del crédito y la cuota con capacidad económica.
    out['loan_to_income'] = (
        out['funded_amnt'] / out['annual_income'].replace(0, np.nan)
    )
    out['installment_to_income'] = (
        out['monthly_payment'] * 12
        / out['annual_income'].replace(0, np.nan)
    )

    # DTI = -1 se conserva como código especial y no como un DTI económico negativo.
    out['dti_special_flag'] = (out['dept_paym_income_ratio'] < 0).astype('int8')
    out['dti_clean'] = out['dept_paym_income_ratio'].where(
        out['dept_paym_income_ratio'] >= 0,
        np.nan
    )
    return out

X_model = construir_features(X)


## 6. PD: WoE + regresión logística

Mantengo WoE + logística para aislar el efecto del margen. La solución de referencia mostró después que LightGBM podía mejorar la discriminación, pero si cambiara simultáneamente el modelo de PD y la política de cobertura ya no podría saber cuánto de la mejora viene realmente del aprendizaje que quiero estudiar.

Los bins se aprenden dentro de Fit. Para el cálculo de los cortes uso una muestra grande y reproducible de Fit por eficiencia computacional; la regresión logística se ajusta sobre todo Fit.


In [6]:
cat_pd = [
    'emp_length', 'home_ownership_status', 'verification_status',
    'loan_purpose', 'initial_list_status', 'disbursement_method',
    'loan_term_months', 'region_code'
]
num_pd = [
    'interest_rate', 'annual_income', 'dti_clean', 'dti_special_flag',
    'used_credit_share', 'num_inq_in_6mths', 'num_30+_delinq_in_2yrs',
    'num_derogatory_pub_rec', 'funded_amnt', 'credit_history_years',
    'loan_to_income', 'installment_to_income', 'issue_date_year'
]

class FastWOE:
    def __init__(self, q=10, min_freq=0.002, smoothing=0.5):
        self.q = q
        self.min_freq = min_freq
        self.smoothing = smoothing
        self.spec = {}
        self.iv_ = {}

    def _calc(self, codes, target, n_codes):
        good = np.bincount(codes[target == 0], minlength=n_codes).astype(float) + self.smoothing
        bad = np.bincount(codes[target == 1], minlength=n_codes).astype(float) + self.smoothing
        dist_good = good / good.sum()
        dist_bad = bad / bad.sum()
        woe = np.log(dist_good / dist_bad)
        iv = np.sum((dist_good - dist_bad) * woe)
        return woe, float(iv)

    def fit(self, data, target):
        target = np.asarray(target, dtype='int8')

        for col in num_pd:
            values = pd.to_numeric(data[col], errors='coerce').to_numpy(float)
            ok = np.isfinite(values)
            valid = values[ok]
            unique = np.sort(np.unique(valid))

            if len(unique) <= 10:
                thresholds = (
                    (unique[:-1] + unique[1:]) / 2
                    if len(unique) > 1 else np.array([])
                )
            else:
                thresholds = np.unique(
                    np.quantile(valid, np.linspace(0, 1, self.q + 1)[1:-1])
                )

            codes = np.digitize(np.where(ok, values, 0), thresholds, right=True)
            missing_code = len(thresholds) + 1
            codes[~ok] = missing_code
            woe, iv = self._calc(codes, target, missing_code + 1)
            self.spec[col] = ('num', thresholds, woe, missing_code)
            self.iv_[col] = iv

        for col in cat_pd:
            s = data[col].astype('object').where(~data[col].isna(), '__MISSING__').astype(str)
            freq = s.value_counts(normalize=True)
            keep = set(freq[freq >= self.min_freq].index)
            s = s.where(s.isin(keep), '__OTHER__')
            levels = sorted(s.unique())
            mapping = {v: i for i, v in enumerate(levels)}
            codes = s.map(mapping).to_numpy(int)
            woe, iv = self._calc(codes, target, len(levels))
            self.spec[col] = ('cat', keep, levels, woe)
            self.iv_[col] = iv
        return self

    def transform(self, data, columns):
        out = np.empty((len(data), len(columns)), dtype='float32')

        for j, col in enumerate(columns):
            spec = self.spec[col]
            if spec[0] == 'num':
                _, thresholds, woe, missing_code = spec
                values = pd.to_numeric(data[col], errors='coerce').to_numpy(float)
                ok = np.isfinite(values)
                codes = np.digitize(np.where(ok, values, 0), thresholds, right=True)
                codes[~ok] = missing_code
                out[:, j] = woe[codes]
            else:
                _, keep, levels, woe = spec
                s = data[col].astype('object').where(~data[col].isna(), '__MISSING__').astype(str)
                s = s.where(s.isin(keep), '__OTHER__')
                codes = pd.Categorical(s, categories=levels).codes
                codes = np.where(codes < 0, 0, codes)
                out[:, j] = woe[codes]
        return out

# Los cortes WoE se aprenden únicamente dentro de Fit.
bin_idx = np.sort(
    rng.choice(fit_idx, size=min(300_000, len(fit_idx)), replace=False)
)
woe = FastWOE().fit(
    X_model.loc[bin_idx, num_pd + cat_pd],
    y.iloc[bin_idx]
)

iv_table = pd.Series(woe.iv_, name='IV').sort_values(ascending=False).to_frame()
selected_pd = iv_table.query('IV >= 0.01').index.tolist()

X_fit_woe = woe.transform(X_model.loc[fit_idx], selected_pd)
X_val_woe = woe.transform(X_model.loc[val_idx], selected_pd)

pd_model = LogisticRegression(
    C=1.0,
    solver='lbfgs',
    max_iter=200,
    random_state=SEED
)
pd_model.fit(X_fit_woe, y.iloc[fit_idx])
pd_val = pd_model.predict_proba(X_val_woe)[:, 1]

pd.DataFrame({
    'métrica': ['AUC', 'Brier', 'PD media', 'Default observado'],
    'valor': [
        roc_auc_score(y.iloc[val_idx], pd_val),
        brier_score_loss(y.iloc[val_idx], pd_val),
        pd_val.mean(),
        y.iloc[val_idx].mean()
    ]
})


,métrica,valor
0,AUC,0.742054
1,Brier,0.103607
2,PD media,0.130740
3,Default observado,0.130685



La PD mantiene una calibración agregada cercana a la tasa de default observada. En la corrida final de esta segunda versión, la validación quedó alrededor de 13.07% de PD media frente a 13.07% de default observado y un AUC cercano a 0.742.

Esto afirma el diagnóstico del feedback en donde podía existir margen para mejorar discriminación con un challenger no lineal, pero no era necesario reemplazar la PD para explicar el exceso de ECL de la V1. La prioridad sigue estando en la política de margen.


## 7. Modelos de EAD y LGD

EAD y LGD se modelizan únicamente en defaults de Fit. Para EAD utilizo la proporción `EAD / funded_amnt` y la LGD se mantiene acotada a [0,1]. Mantengo árboles poco profundos para capturar relaciones no lineales sin aumentar complejidad de forma innecesaria.


In [7]:
component_cat = [
    'emp_length', 'home_ownership_status', 'verification_status',
    'loan_purpose', 'initial_list_status', 'disbursement_method',
    'loan_term_months'
]
component_num = [
    'funded_amnt', 'interest_rate', 'monthly_payment', 'annual_income',
    'dti_clean', 'used_credit_share', 'num_inq_in_6mths',
    'num_30+_delinq_in_2yrs', 'total_credit_revolving_bal',
    'issue_date_year', 'credit_history_years', 'loan_to_income',
    'installment_to_income'
]

X_component = X_model[component_cat + component_num].copy()
component_levels = {}

# La codificación de categorías se aprende solo en Fit.
for col in component_cat:
    fit_s = X_component.loc[fit_idx, col].astype('object').where(
        ~X_component.loc[fit_idx, col].isna(), '__MISSING__'
    ).astype(str)
    levels = sorted(fit_s.unique())
    mapping = {v: i for i, v in enumerate(levels)}
    component_levels[col] = mapping

    all_s = X_component[col].astype('object').where(
        ~X_component[col].isna(), '__MISSING__'
    ).astype(str)
    X_component[col] = all_s.map(mapping).fillna(-1).astype('int16')

fit_defaults = fit_idx[y.iloc[fit_idx].to_numpy() == 1]

component_params = dict(
    n_estimators=160,
    learning_rate=0.04,
    num_leaves=15,
    max_depth=4,
    min_child_samples=400,
    subsample=0.85,
    colsample_bytree=0.90,
    reg_lambda=3.0,
    reg_alpha=0.3,
    random_state=SEED,
    n_jobs=4,
    verbosity=-1
)

model_ead = LGBMRegressor(**component_params)
model_ead.fit(
    X_component.loc[fit_defaults],
    ead_ratio.iloc[fit_defaults]
)

fit_lgd = np.array([
    i for i in fit_defaults if pd.notna(lgd_obs.iloc[i])
])
model_lgd = LGBMRegressor(**component_params)
model_lgd.fit(
    X_component.loc[fit_lgd],
    lgd_obs.iloc[fit_lgd]
)


,boosting_type,'gbdt'
,num_leaves,15
,max_depth,4
,learning_rate,0.04
,n_estimators,160
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,400


## 8. ECL basal y métricas

Antes de introducir margen reviso si los tres componentes ya reproducen razonablemente la pérdida agregada. Este bloque es importante porque separa dos problemas distintos:

1. **Calibración agregada:** cuánto suma el ECL frente a la pérdida real.
2. **Distribución individual:** en cuántas operaciones el ECL queda por debajo de la pérdida realizada.

También separo MAPE de desviación agregada. El MAPE se calcula únicamente sobre observaciones con pérdida real positiva y la desviación agregada compara los dos totales de cartera.


In [8]:
pred_ead_ratio_val = np.clip(
    model_ead.predict(X_component.loc[val_idx]), 0, 1
)
pred_lgd_val = np.clip(
    model_lgd.predict(X_component.loc[val_idx]), 0, 1
)
funded_val = X.loc[val_idx, 'funded_amnt'].to_numpy(float)
pred_ead_val = funded_val * pred_ead_ratio_val

ecl_base_val = pd_val * pred_lgd_val * pred_ead_val
loss_val = loss_real[val_idx]

def metricas_ecl(pred, real):
    pred = np.asarray(pred, dtype=float)
    real = np.asarray(real, dtype=float)
    error = np.abs(pred - real)
    con_perdida = real >= 1

    return {
        'ECL total': float(pred.sum()),
        'Pérdida real': float(real.sum()),
        'Ratio ECL/real': float(pred.sum() / real.sum()),
        '% ECL < pérdida': float(100 * np.mean(pred < real)),
        'MAE': float(error.mean()),
        'MAPE (%)': float(100 * np.mean(error[con_perdida] / real[con_perdida])),
        'Desviación agregada (%)': float(
            100 * abs(pred.sum() - real.sum()) / real.sum()
        )
    }

pd.Series(metricas_ecl(ecl_base_val, loss_val)).to_frame('valor')


,valor
ECL total,4.445825e+08
Pérdida real,4.459826e+08
Ratio ECL/real,9.968607e-01
% ECL < pérdida,1.280383e+01
MAE,2.033037e+03
MAPE (%),8.374719e+01
Desviación agregada (%),3.139287e-01


El patrón se mantiene, en donde el ECL basal prácticamente coincide con la pérdida real agregada, pero alrededor de 12.8% de la cartera queda por debajo de su pérdida real. Esto cambia la pregunta del problema. Ya no necesito “subir toda la cartera”. Necesito encontrar qué operaciones conviene cubrir y cuánto cuesta llevarlas al nivel necesario.


## 9. V2A: Margen por costo incremental

La primera mejora utiliza la pérdida condicionada estimada:

$$L_i^{cond} = LGD_i \times EAD_i$$

pero la solución de referencia me permitió afinar una idea adicional. Cubrir una operación no cuesta todo `LGD × EAD`, porque el préstamo ya tiene una provisión basal:

$$ECL_i^{base} = PD_i \times LGD_i \times EAD_i$$

Si el piso es $m \times LGD_i \times EAD_i$, el costo que realmente agrego es:

$$Costo_i^{inc} = \max(m \times LGD_i \times EAD_i - ECL_i^{base},\; \varepsilon)$$

Por eso ordeno por:

$$Score_i = \frac{PD_i}{Costo_i^{inc}}$$

Se debe priorizar operaciones con mayor probabilidad de default por cada dólar adicional que necesito agregar para cubrirlas.


In [9]:
# Divido Validación en dos mitades. La primera ajusta el margen y la segunda comprueba que la regla funcione sin modificar parámetros.

cal_pos, check_pos = train_test_split(
    np.arange(len(val_idx)),
    test_size=0.50,
    random_state=SEED
)
cal_pos = np.asarray(cal_pos)
check_pos = np.asarray(check_pos)

loss_cond_val = pred_lgd_val * pred_ead_val

def top_q_mask(score, q):
    """Marca exactamente la fracción q con mayor puntuación."""
    n = len(score)
    k = max(1, min(n, int(np.ceil(q * n))))
    order = np.argsort(-score)
    mask = np.zeros(n, dtype=bool)
    mask[order[:k]] = True
    return mask


def aplicar_v2a(base, pd_pred, loss_cond, funded, q, m):
    # Cobertura que recibiría cada préstamo si entra en el tramo seleccionado.
    covered = np.maximum(
        base,
        np.minimum(m * loss_cond, funded)
    )

    # Costo incremental: cuánto ECL adicional tengo que agregar sobre el basal.
    incremental_cost = np.maximum(covered - base, 1e-9)
    score = pd_pred / incremental_cost
    mask = top_q_mask(score, q)

    out = base.copy()
    out[mask] = covered[mask]
    return out, mask


### 9.1. ¿Cuánto colchón dejo antes de abrir Test?

En la V1 elegí un margen fijo muy cercano al límite del 10%. En esta versión quiero que el colchón tenga una justificación observable.

Para cada objetivo de subestimación pruebo combinaciones de `q` y `m` en la mitad de calibración. Después aplico la regla congelada a la mitad de comprobación y realizo 300 muestras bootstrap del tamaño del Test. No me quedo con el objetivo más agresivo. Exijo que el peor porcentaje observado en bootstrap permanezca por debajo de 9.80%. Este margen no garantiza el futuro, pero evita diseñar la política exactamente contra el límite reglado del ejercicio.


In [11]:
def bootstrap_stats(pred, real, sample_size, n_boot=N_BOOT):
    pred = np.asarray(pred, dtype=float)
    real = np.asarray(real, dtype=float)
    n = len(real)

    pct_below = np.empty(n_boot)
    ratio_total = np.empty(n_boot)

    for b in range(n_boot):
        idx = rng.integers(0, n, size=sample_size)
        p = pred[idx]
        r = real[idx]
        pct_below[b] = 100 * np.mean(p < r)
        ratio_total[b] = p.sum() / r.sum()

    return pct_below, ratio_total


def buscar_politica_v2a(pos, objetivos=(9.2, 9.3, 9.4, 9.5, 9.6, 9.7, 9.8)):
    base = ecl_base_val[pos]
    pd_pred = pd_val[pos]
    loss_cond = loss_cond_val[pos]
    funded = funded_val[pos]
    real = loss_val[pos]
    n = len(pos)

    base_under = (base < real).astype(np.int8)
    base_under_total = base_under.sum()
    base_total = base.sum()
    real_total = real.sum()

    best = {obj: None for obj in objetivos}

    for m in np.arange(0.80, 2.001, 0.025):
        covered = np.maximum(base, np.minimum(m * loss_cond, funded))
        delta = covered - base
        score = pd_pred / np.maximum(delta, 1e-9)
        order = np.argsort(-score)

        # Acumulados: evitan recalcular toda la cartera para cada q.
        delta_cum = np.cumsum(delta[order])
        under_change = (
            (covered < real).astype(np.int8) - base_under
        )[order]
        under_cum = np.cumsum(under_change)

        for q in np.arange(0.05, 0.401, 0.005):
            k = max(1, min(n, int(np.ceil(q * n))))
            total = base_total + delta_cum[k - 1]
            under = 100 * (base_under_total + under_cum[k - 1]) / n

            if total < real_total:
                continue

            for objetivo in objetivos:
                if under <= objetivo:
                    candidate = {
                        'q': float(q),
                        'm': float(m),
                        'ECL_cal': float(total),
                        'under_cal': float(under)
                    }
                    if best[objetivo] is None or total < best[objetivo]['ECL_cal']:
                        best[objetivo] = candidate
    return best

candidatos = buscar_politica_v2a(cal_pos)
revision = []

for objetivo, params in candidatos.items():
    if params is None:
        continue

    pred_check, _ = aplicar_v2a(
        ecl_base_val[check_pos],
        pd_val[check_pos],
        loss_cond_val[check_pos],
        funded_val[check_pos],
        params['q'],
        params['m']
    )
    met = metricas_ecl(pred_check, loss_val[check_pos])
    pct_boot, ratio_boot = bootstrap_stats(
        pred_check,
        loss_val[check_pos],
        sample_size=len(test_idx)
    )

    revision.append({
        'objetivo': objetivo,
        **params,
        'under_check': met['% ECL < pérdida'],
        'ratio_check': met['Ratio ECL/real'],
        'bootstrap_max_under': pct_boot.max(),
        'bootstrap_min_ratio': ratio_boot.min()
    })

revision = pd.DataFrame(revision)
revision


,objetivo,q,m,ECL_cal,under_cal,under_check,ratio_check,bootstrap_max_under,bootstrap_min_ratio
0,9.2,0.175,1.550,3.741153e+08,9.186136,9.377702,1.662981,9.534993,1.638790
1,9.3,0.170,1.550,3.677416e+08,9.286036,9.479938,1.634616,9.642020,1.613763
2,9.4,0.165,1.550,3.613598e+08,9.391194,9.583343,1.606665,9.727315,1.583710
3,9.5,0.155,1.625,3.549776e+08,9.494599,9.677401,1.578820,9.807935,1.553039
4,9.6,0.150,1.650,3.504868e+08,9.569962,9.740495,1.559107,9.901409,1.534522
5,9.7,0.145,1.650,3.443349e+08,9.687973,9.849742,1.531790,9.962400,1.513819
6,9.8,0.145,1.575,3.390078e+08,9.764505,9.942047,1.507865,10.064520,1.485577


### 9.2. Decisión V2A

La regla robusta selecciona un objetivo de 9.4% en calibración. En la corrida final, esto corresponde aproximadamente a:

- cubrir el 16.5% de la cartera con mayor score de costo incremental
- utilizar un piso de 1.55 × LGD × EAD
- peor porcentaje por debajo en las 300 muestras bootstrap de aproximadamente 9.70%.

El punto importante no es memorizar esos parámetros. Si cambia la semilla o los modelos, el código vuelve a aprenderlos. Lo que permanece es la lógica de decisión.


## 10. V2B: Experimento adicional con $\kappa$

Después de construir V2A revisé las soluciones compartidas por otros participantes y, posteriormente, la solución de referencia del reto. Una idea que aparecía en las mejores estrategias era reducir el ECL basal del tramo que no recibe cobertura mediante un factor $\kappa$:

$$ECL_i^{V2B} = \kappa ECL_i^{base}, \quad i \notin cobertura$$

No considero esta idea parte de mi solución original. La incluyo como experimento porque ayuda a entender qué está premiando exactamente la función de evaluación.

Puede funcionar muy bien en este reto porque los préstamos sanos tienen pérdida real igual a cero y las restricciones se concentran en dos puntos: que el ECL total supere la pérdida real y que no más del 10% de las operaciones quede por debajo.

No trasladaría esta regla directamente a una provisión bancaria. En un entorno real tendría que justificar el nivel de provisión por riesgo, estabilidad, política contable, validación independiente y requerimientos regulatorios. Aquí el objetivo es entender el comportamiento de la métrica, no convertir $\kappa$ en una recomendación prudencial.

In [12]:
def calibrar_kappa(base, pd_pred, loss_cond, funded, real, q, m, max_under=9.60):
    # Mantengo q y m de la política principal; solo estudio cuánto podría reducirse el ECL basal del tramo no cubierto sin romper las reglas.
    covered = np.maximum(base, np.minimum(m * loss_cond, funded))
    score = pd_pred / np.maximum(covered - base, 1e-9)
    mask = top_q_mask(score, q)

    best = None
    for kappa in np.arange(0.05, 1.001, 0.005):
        pred = kappa * base
        pred[mask] = covered[mask]
        met = metricas_ecl(pred, real)

        if (
            met['Ratio ECL/real'] >= 1
            and met['% ECL < pérdida'] <= max_under
        ):
            if best is None or met['ECL total'] < best['metrics']['ECL total']:
                best = {'kappa': float(kappa), 'metrics': met}
    return best

q_v2a = 0.165
m_v2a = 1.55

# 1) k se aprende en la mitad de calibración.
best_kappa_cal = calibrar_kappa(
    ecl_base_val[cal_pos],
    pd_val[cal_pos],
    loss_cond_val[cal_pos],
    funded_val[cal_pos],
    loss_val[cal_pos],
    q_v2a,
    m_v2a,
    max_under=9.60
)
kappa_cal = best_kappa_cal['kappa']

# 2) Compruebo que generalice en la segunda mitad de validación.
base_check = ecl_base_val[check_pos]
covered_check = np.maximum(
    base_check,
    np.minimum(m_v2a * loss_cond_val[check_pos], funded_val[check_pos])
)
score_check = pd_val[check_pos] / np.maximum(covered_check - base_check, 1e-9)
mask_check = top_q_mask(score_check, q_v2a)

kappa_check = None
for candidate_k in np.arange(kappa_cal, 1.001, 0.005):
    pred_check = candidate_k * base_check
    pred_check[mask_check] = covered_check[mask_check]
    met_check = metricas_ecl(pred_check, loss_val[check_pos])

    if (
        met_check['Ratio ECL/real'] >= 1
        and met_check['% ECL < pérdida'] <= 9.70
    ):
        kappa_check = float(candidate_k)
        break

# 3) Uso bootstrap para no dejar el total agregado pegado al límite. 
# Para cada remuestra calculo qué κ mínimo habría hecho falta para que el ECL total no quedara por debajo de la pérdida real.
high_contrib = np.zeros_like(base_check)
high_contrib[mask_check] = covered_check[mask_check]
low_base = base_check.copy()
low_base[mask_check] = 0.0

kappa_required = []
bootstrap_indices = []
for _ in range(N_BOOT):
    idx_boot = rng.integers(0, len(check_pos), size=len(test_idx))
    bootstrap_indices.append(idx_boot)

    real_total_boot = loss_val[check_pos][idx_boot].sum()
    high_total_boot = high_contrib[idx_boot].sum()
    low_base_boot = low_base[idx_boot].sum()

    k_req = max(
        0.0,
        (real_total_boot - high_total_boot) / max(low_base_boot, 1e-9)
    )
    kappa_required.append(k_req)

# Redondeo hacia arriba a la misma malla de 0.005 utilizada en calibración.
kappa_boot = np.ceil(max(kappa_required) / 0.005) * 0.005
kappa = float(max(kappa_check, kappa_boot))

# Verificación final de estabilidad en las 300 remuestras.
pred_check = kappa * base_check
pred_check[mask_check] = covered_check[mask_check]
boot_under = []
boot_ratio = []
for idx_boot in bootstrap_indices:
    p = pred_check[idx_boot]
    r = loss_val[check_pos][idx_boot]
    boot_under.append(100 * np.mean(p < r))
    boot_ratio.append(p.sum() / r.sum())

print(f'κ calibración: {kappa_cal:.3f}')
print(f'κ validación: {kappa_check:.3f}')
print(f'κ requerido bootstrap: {kappa_boot:.3f}')
print(f'κ final: {kappa:.3f}')
print(f'Máx. % debajo bootstrap: {max(boot_under):.2f}%')
print(f'Mín. ratio bootstrap: {min(boot_ratio):.4f}x')


κ calibración: 0.270
κ validación: 0.290
κ requerido bootstrap: 0.310
κ final: 0.310
Máx. % debajo bootstrap: 9.80%
Mín. ratio bootstrap: 1.0026x


### 10.1. Decisión V2B

La calibración termina en **$\kappa = 0.31$**. 
El valor no sale de Test, primero se aprende en calibración, después se comprueba en la segunda mitad de Validación y finalmente se eleva lo necesario para que el ECL agregado no quede por debajo de la pérdida real en ninguna de las 300 remuestras bootstrap utilizadas como control de estabilidad.

Esta variante reduce de forma fuerte el ECL respecto a V2A y también mejora el MAE. El MAPE, sin embargo, es mayor que en el ECL basal y que en mi entrega original. Esto confirma que $\kappa$ está optimizando las reglas específicas del reto y no todas las dimensiones de error al mismo tiempo.

El ECL de esta variante queda muy cerca del resultado obtenido por la solución de referencia. Esa cercanía no es una coincidencia ni una réplica exacta. Una vez que distintas soluciones utilizan el mismo split, mantienen el total apenas por encima de la pérdida real y aplican una reducción tipo $\kappa$ al tramo no cubierto, es razonable que tiendan a converger a una zona parecida. La diferencia está en cómo se modelan PD, LGD y EAD, cómo se selecciona la cobertura y cómo se calibra el margen.

Por esa razón mantengo la jerarquía del análisis en donde V2A es mi propuesta principal del reto (post-análisis) y V2B es un experimento adicional informado por los aprendizajes publicados después del reto.

## 11. Evaluación final: Con Test

Hasta esta sección Test no ha participado en ninguna decisión. Ahora aplico los modelos y las dos políticas ya congeladas. La pérdida real se utiliza únicamente para medir el resultado final.


In [13]:
# PD en Test: WoE y regresión ya aprendidos en Fit.
X_test_woe = woe.transform(X_model.loc[test_idx], selected_pd)
pd_test = pd_model.predict_proba(X_test_woe)[:, 1]

# EAD y LGD con los modelos congelados.
pred_ead_ratio_test = np.clip(
    model_ead.predict(X_component.loc[test_idx]), 0, 1
)
pred_lgd_test = np.clip(
    model_lgd.predict(X_component.loc[test_idx]), 0, 1
)
funded_test = X.loc[test_idx, 'funded_amnt'].to_numpy(float)
pred_ead_test = funded_test * pred_ead_ratio_test
loss_cond_test = pred_lgd_test * pred_ead_test

ecl_base_test = pd_test * loss_cond_test
loss_real_test = loss_real[test_idx]

# V2A: top q por costo incremental.
pe_v2a, mask_v2a = aplicar_v2a(
    ecl_base_test,
    pd_test,
    loss_cond_test,
    funded_test,
    q_v2a,
    m_v2a
)

# V2B: misma cobertura, pero κ en el tramo no cubierto.
covered_test = np.maximum(
    ecl_base_test,
    np.minimum(m_v2a * loss_cond_test, funded_test)
)
score_test = pd_test / np.maximum(covered_test - ecl_base_test, 1e-9)
mask_v2b = top_q_mask(score_test, q_v2a)
pe_v2b = kappa * ecl_base_test
pe_v2b[mask_v2b] = covered_test[mask_v2b]

comparison = pd.DataFrame([
    {'Estrategia': 'ECL basal', **metricas_ecl(ecl_base_test, loss_real_test)},
    {'Estrategia': 'V2A · costo incremental + bootstrap', **metricas_ecl(pe_v2a, loss_real_test)},
    {'Estrategia': 'V2B · V2A + κ', **metricas_ecl(pe_v2b, loss_real_test)},
])

comparison


,Estrategia,ECL total,Pérdida real,Ratio ECL/real,% ECL < pérdida,MAE,MAPE (%),Desviación agregada (%)
0,ECL basal,5.552133e+08,5.549873e+08,1.000407,12.830867,2028.450054,83.004754,0.040714
1,V2A · costo incremental + bootstrap,9.021300e+08,5.549873e+08,1.625497,9.487321,2584.030023,91.710345,62.549658
2,V2B · V2A + κ,5.729691e+08,5.549873e+08,1.032400,9.580094,2125.438247,97.522886,3.240030


### 11.1. Resultados

Con el split oficial y los modelos completos, los resultados obtenidos fueron:

| Estrategia | ECL total | ECL / pérdida real | % por debajo | MAE | MAPE |
|---|---:|---:|---:|---:|---:|
| ECL basal | USD 555.2 M | 1.0004x | 12.83% | USD 2,028 | 83.00% |
| **V2A · propuesta principal: costo incremental + bootstrap** | **USD 902.1 M** | **1.6255x** | **9.49%** | **USD 2,584** | **91.71%** |
| V2B · experimento adicional con $\kappa$ | USD 573.0 M | 1.0324x | 9.58% | USD 2,125 | 97.52% |

Con los resultados de V2A frente a mi entrega original, el ratio ECL/pérdida real baja de aproximadamente 1.90x a 1.63x sin reducir el ECL basal de las operaciones que no reciben cobertura. La mejora viene de cambiar dos decisiones, primero, cuánto necesita realmente cada préstamo para quedar cubierto, y segundo, en qué operaciones conviene gastar primero el margen.

Pero V2B lleva el ECL hasta aproximadamente 1.03x y reduce el MAE frente a V2A, pero el MAPE aumenta. No lo interpreto como una estrategia superior en todas las dimensiones. Es una demostración de que una función objetivo puede premiar una solución distinta a la que elegiría si el criterio central fuera el error porcentual o si estuviera definiendo una política prudencial para una entidad financiera.

Además, V2B fue construido después de conocer el feedback y estudiar las ideas compartidas por otros cracks de la comunidadad. Por eso no lo presento como un resultado que habría obtenido durante la competencia ni como una solución desarrollada de forma independiente.

## 12. ¿Qué aprendí al contrastar con la solución de referencia?

Después de construir esta V2 revisé la solución de Jyts. Hay tres ideas que considero relevantes, pero no las incorporo retrospectivamente como si fueran parte de mi segunda estrategia:

**1. LightGBM como challenger de PD.** Se obtiene mayor discriminación que la logística manteniendo buena calibración. En una siguiente iteración lo probaría como challenger y mediría si la mejora de AUC se traduce realmente en menor costo de margen.

**2. Cuantiles de pérdida.** En lugar de cubrir a todos con un múltiplo de la pérdida media estimada, se puede modelar directamente un cuantil de la pérdida. Eso permite adaptar el colchón a la incertidumbre de cada operación. Es probablemente la siguiente mejora más interesante para LGD × EAD.

**3. Censura y madurez.** Mi EDA ya había detectado diferencias por cosecha, las variables de madurez pueden aprovechar esa señal. La incorporaría con cuidado, siempre verificando que pueda construirse con información disponible al momento correspondiente.

No agrego estas tres mejoras al resultado principal porque el objetivo de este análisis es aislar el aprendizaje del margen. Si cambiara todos los componentes al mismo tiempo, perdería trazabilidad sobre qué decisión produjo cada mejora.

## 13. ¿Qué me llevo del reto?

El aprendizaje principal no fue “usar un modelo más complejo”. Fue entender que un modelo puede estar bien calibrado y aun así resolver de forma ineficiente la decisión que viene después.

En mi primera entrega pregunté principalmente ¿qué préstamos tenían mayor PD?. Después del feedback cambié la pregunta a ¿cuánto cuesta cubrir cada operación y cuánto riesgo reduzco por cada dólar adicional de ECL?. Esa reformulación explica una parte importante de la mejora.

Me llevo cinco criterios que quiero conservar para próximos proyectos de Data Science:

1. **Entender la métrica antes de optimizarla.** Una buena predicción no garantiza una buena decisión si la función objetivo mide otra cosa.
2. **Separar modelado y política de decisión.** PD, LGD y EAD pueden estar razonablemente calibrados y aun así el margen final ser ineficiente.
3. **No fijar un colchón exactamente contra un límite.** Si existe una restricción, hay que revisar su estabilidad y no solo el punto estimado. Aquí el bootstrap fue útil para eso.
4. **Comparar trade-offs, no un solo número.** ECL total, MAE, MAPE y porcentaje por debajo pueden moverse en direcciones distintas.
5. **Distinguir una mejora de riesgo de una optimización de competencia.** Que una regla mejore el score del reto no significa que deba implementarse igual en banca.

Si tuviera que defender una estrategia en un contexto bancario, me quedaría con la lógica de V2A y seguiría trabajando la estimación de severidad y la incertidumbre de la pérdida. V2B me parece valiosa como aprendizaje de optimización, pero necesitaría una justificación completamente distinta antes de trasladarla a provisiones reales.

## 14. Aprendizajes al contrastar con otras soluciones

Para esta segunda versión contrasté mi solución con el feedback individual del reto, con los notebooks compartidos posteriormente por otros participantes y con la solución de referencia. De esa comparación tomé ideas conceptuales para cuestionar mi política de margen para la cobertura basada en `LGD × EAD`, costo incremental, bootstrap, $\kappa$ y modelización cuantílica de la pérdida.

No intento reproducir esos pipelines completos. Mantengo mi arquitectura base para poder identificar qué parte de la mejora viene de cambiar la política de margen. Dos ideas quedan como challengers para una siguiente iteración, no como resultados propios de esta V2:

- **LightGBM para PD**, porque la solución de referencia mostró una mejora de discriminación manteniendo una calibración adecuada.
- **Regresión cuantílica de la pérdida**, porque permite modelar directamente la incertidumbre de la severidad en lugar de utilizar el mismo múltiplo de la pérdida media para todos los préstamos.

Esta separación también es parte del aprendizaje, porque en un proyecto de portfolio prefiero dejar claro qué desarrollé, qué adapté después del feedback y qué aprendí revisando el trabajo de otros.

## 15. Controles y salida

Antes del cierre verifico que Test sea disjunto de Fit/Validación y que las predicciones sean válidas. Estos controles no modifican ningún resultado.


In [14]:
# Test no puede haber participado en Fit ni Validación.
assert set(fit_idx).isdisjoint(set(test_idx))
assert set(val_idx).isdisjoint(set(test_idx))

# Las predicciones finales deben existir para toda la muestra y ser no negativas.
for arr in [ecl_base_test, pe_v2a, pe_v2b]:
    assert len(arr) == len(test_idx)
    assert np.isfinite(arr).all()
    assert (arr >= 0).all()

# Para portfolio guardo ambas variantes, pero V2A queda identificada como la propuesta principal. 
# V2B se conserva únicamente como experimento adicional.
resultado_postchallenge = pd.DataFrame({
    'PE_V2A_PRINCIPAL': pe_v2a,
    'PE_V2B_EXPERIMENTO_KAPPA': pe_v2b
})
resultado_postchallenge.to_csv(
    'resultado_postchallenge_ecl_v2.csv',
    index=False
)

# Si necesitara un archivo con una sola PE como salida principal del proyecto, utilizaría V2A.
pd.DataFrame({'PE': pe_v2a}).to_csv(
    'resultado_postchallenge_v2a_principal.csv',
    index=False
)

print('Validaciones finales superadas.')

Validaciones finales superadas.


In [15]:
# Última celda: únicamente resume las métricas del reto.
# A partir de aquí no se entrena, calibra ni modifica ninguna predicción.
res_v2a = metricas_ecl(pe_v2a, loss_real_test)
res_v2b = metricas_ecl(pe_v2b, loss_real_test)

print('=' * 76)
print('RESULTADO POST-CHALLENGE — RETO 01 ECL')
print('=' * 76)
print(f'SEED oficial                : {SEED}')
print(f'Pérdida real Test           : ${res_v2a["Pérdida real"]:,.2f}')
print('-' * 76)
print('V2A · PROPUESTA PRINCIPAL POST-CHALLENGE')
print('     Costo incremental + bootstrap; sin reducir ECL del tramo no cubierto')
print(f'ECL total                   : ${res_v2a["ECL total"]:,.2f}')
print(f'ECL / pérdida real          : {res_v2a["Ratio ECL/real"]:.4f}x')
print(f'% ECL < pérdida             : {res_v2a["% ECL < pérdida"]:.2f}%')
print(f'MAE                         : ${res_v2a["MAE"]:,.2f}')
print(f'MAPE                        : {res_v2a["MAPE (%)"]:.2f}%')
print(f'Restricciones               : {"CUMPLE" if res_v2a["Ratio ECL/real"] > 1 and res_v2a["% ECL < pérdida"] <= 10 else "NO CUMPLE"}')
print('-' * 76)
print('V2B · EXPERIMENTO ADICIONAL CON κ')
print('     Incorporado después de revisar soluciones compartidas post-reto')
print(f'ECL total                   : ${res_v2b["ECL total"]:,.2f}')
print(f'ECL / pérdida real          : {res_v2b["Ratio ECL/real"]:.4f}x')
print(f'% ECL < pérdida             : {res_v2b["% ECL < pérdida"]:.2f}%')
print(f'MAE                         : ${res_v2b["MAE"]:,.2f}')
print(f'MAPE                        : {res_v2b["MAPE (%)"]:.2f}%')
print(f'Restricciones               : {"CUMPLE" if res_v2b["Ratio ECL/real"] > 1 and res_v2b["% ECL < pérdida"] <= 10 else "NO CUMPLE"}')
print('-' * 76)
print('Lectura final: V2A es la propuesta principal del portfolio.')
print('V2B se conserva como aprendizaje de optimización de la métrica del reto.')
print('=' * 76)

RESULTADO POST-CHALLENGE — RETO 01 ECL
SEED oficial                : 1710
Pérdida real Test           : $554,987,330.99
----------------------------------------------------------------------------
V2A · PROPUESTA PRINCIPAL POST-CHALLENGE
     Costo incremental + bootstrap; sin reducir ECL del tramo no cubierto
ECL total                   : $902,130,008.85
ECL / pérdida real          : 1.6255x
% ECL < pérdida             : 9.49%
MAE                         : $2,584.03
MAPE                        : 91.71%
Restricciones               : CUMPLE
----------------------------------------------------------------------------
V2B · EXPERIMENTO ADICIONAL CON κ
     Incorporado después de revisar soluciones compartidas post-reto
ECL total                   : $572,969,086.43
ECL / pérdida real          : 1.0324x
% ECL < pérdida             : 9.58%
MAE                         : $2,125.44
MAPE                        : 97.52%
Restricciones               : CUMPLE
----------------------------------------